# OpenAI Function Calling 입문 실습 (Colab)

이 노트북은 **OpenAI 공식 Function Calling** 흐름을 가장 작은 예제로 보여줍니다.

## 이 실습에서 배우는 것

- OpenAI **Responses API**에서 `tools`로 function을 정의하는 방법
- 모델이 `function_call`을 반환하면, 애플리케이션 코드가 실제 함수를 실행하는 흐름
- `function_call_output`을 다시 모델에 보내 **최종 답변**을 받는 방법
- 왜 이 구조가 나중에 **MCP tool 호출 구조와 연결되는지**

## 공식 흐름 요약

OpenAI Function Calling의 기본 흐름은 아래와 같습니다.

1. 모델에 `tools`를 포함해 요청
2. 모델이 `function_call` 반환
3. 애플리케이션이 실제 Python 함수를 실행
4. 그 결과를 `function_call_output`으로 다시 모델에 전달
5. 모델이 최종 답변 생성


## 실행 순서

1. **OpenAI SDK 설치**
2. **API Key 설정**
3. **로컬 Python 함수 정의**
4. **OpenAI tool schema 정의**
5. **첫 번째 요청 보내기**
6. **모델의 function_call 처리**
7. **function_call_output 전송**
8. **최종 답변 확인**
9. **두 번째 예시 실행**

권장:
- 메뉴에서 **런타임 → 모두 실행**
- 또는 위에서 아래로 한 셀씩 실행


In [ ]:
# 1) 설치
!pip -q install -U openai


In [ ]:
# 2) API Key 설정
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

print("OPENAI_API_KEY 설정 완료")


## 1) 로컬 Python 함수 정의

여기서는 두 개의 로컬 함수를 사용합니다.

1. `get_weather(city)`
   - 도시 이름을 받아 날씨 설명 반환

2. `write_opening_line(weather_description, genre)`
   - 날씨 설명과 장르를 받아 소설 첫 문장 생성

중요한 점:
- 모델이 직접 이 함수를 실행하는 것이 아닙니다.
- 모델은 **“이 함수를 호출해 달라”**고 요청만 합니다.
- 실제 실행은 **우리 애플리케이션 코드**가 합니다.


In [ ]:
import json
from openai import OpenAI

client = OpenAI()

def get_weather(city: str) -> str:
    weather_map = {
        "서울": "서울은 화창하고 벚꽃이 피기 좋은 날씨입니다.",
        "부산": "부산은 바닷바람이 부드럽고 맑은 날씨입니다.",
        "제주": "제주는 옅은 안개와 함께 감성적인 새벽 분위기입니다.",
    }
    return weather_map.get(city, f"{city}의 날씨는 소설 배경으로 쓰기 좋은 분위기입니다.")

def write_opening_line(weather_description: str, genre: str) -> str:
    if genre == "romance":
        return f"{weather_description} 그녀를 처음 마주한 순간, 계절은 비밀처럼 천천히 피어나고 있었다."
    elif genre == "mystery":
        return f"{weather_description} 그날의 공기에는 설명할 수 없는 불길한 침묵이 스며 있었다."
    else:
        return f"{weather_description} 그 풍경은 새로운 이야기가 시작되기에 충분히 아름다웠다."

print("로컬 함수 정의 완료")


## 2) OpenAI tool schema 정의

OpenAI Function Calling에서는 각 함수에 대해 **JSON Schema 기반 정의**를 `tools`에 넣어 줍니다.
모델은 이 schema를 보고:
- 어떤 함수가 있는지
- 언제 써야 하는지
- 어떤 인자가 필요한지
를 이해합니다.


In [ ]:
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "도시 이름을 받아 현재 날씨 분위기를 반환합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "날씨를 알고 싶은 도시 이름"
                }
            },
            "required": ["city"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "write_opening_line",
        "description": "날씨 설명과 장르를 받아 소설 첫 문장을 한 줄 생성합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "weather_description": {
                    "type": "string",
                    "description": "날씨/분위기 설명"
                },
                "genre": {
                    "type": "string",
                    "description": "소설 장르",
                    "enum": ["romance", "mystery", "general"]
                }
            },
            "required": ["weather_description", "genre"],
            "additionalProperties": False
        },
        "strict": True
    }
]

print("tool schema 정의 완료")
print(json.dumps(tools, ensure_ascii=False, indent=2))


## 3) Function call 실행 헬퍼

모델이 반환한 `function_call`을 실제 Python 함수에 라우팅하는 헬퍼입니다.


In [ ]:
def call_function(name: str, args: dict):
    if name == "get_weather":
        return get_weather(**args)
    elif name == "write_opening_line":
        return write_opening_line(**args)
    else:
        raise ValueError(f"Unknown function: {name}")


## 4) 첫 번째 요청

질문:
> 지금 서울 날씨를 참고해서, 로맨스 소설의 첫 문장을 한 줄 써줘.

이 질문에 대해 모델은 보통:
1. `get_weather(city="서울")`
2. `write_opening_line(weather_description=..., genre="romance")`
같은 function call을 연속으로 만들 수 있습니다.


In [ ]:
user_input = "지금 서울 날씨를 참고해서, 로맨스 소설의 첫 문장을 한 줄 써줘."

response = client.responses.create(
    model="gpt-4.1",
    input=user_input,
    tools=tools,
    tool_choice="auto",
)

print("첫 번째 응답 생성 완료")
print("response id:", response.id)
print("output item count:", len(response.output))
for item in response.output:
    print({
        "type": getattr(item, "type", None),
        "name": getattr(item, "name", None),
        "call_id": getattr(item, "call_id", None),
        "arguments": getattr(item, "arguments", None),
    })


## 5) 모델의 function_call 처리

공식 흐름에 맞게, `response.output` 안에서 `type == "function_call"`인 항목을 찾아
실제 함수를 실행하고 결과를 `function_call_output`으로 만듭니다.


In [ ]:
function_outputs = []

for item in response.output:
    if getattr(item, "type", None) != "function_call":
        continue

    function_name = item.name
    arguments = json.loads(item.arguments)
    result = call_function(function_name, arguments)

    print(f"함수 실행: {function_name}({arguments}) -> {result}")

    function_outputs.append(
        {
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps({"result": result}, ensure_ascii=False)
        }
    )

print("\nfunction_call_output payload:")
print(json.dumps(function_outputs, ensure_ascii=False, indent=2))


## 6) function_call_output을 다시 모델에 전달

이제 함수 실행 결과를 다시 모델에 주면, 모델이 이를 반영해 최종 답변을 생성합니다.
여러 function call이 있었다면 모두 결과를 묶어서 전달하면 됩니다.


In [ ]:
final_response = client.responses.create(
    model="gpt-4.1",
    previous_response_id=response.id,
    input=function_outputs,
    tools=tools,
)

print("=== 최종 답변 ===")
print(final_response.output_text)


## 7) 두 번째 예시 실행

질문:
> 제주 날씨를 참고해서 미스터리 소설 첫 문장을 한 줄 써줘.


In [ ]:
user_input2 = "제주 날씨를 참고해서 미스터리 소설 첫 문장을 한 줄 써줘."

response2 = client.responses.create(
    model="gpt-4.1",
    input=user_input2,
    tools=tools,
    tool_choice="auto",
)

function_outputs2 = []

for item in response2.output:
    if getattr(item, "type", None) != "function_call":
        continue

    function_name = item.name
    arguments = json.loads(item.arguments)
    result = call_function(function_name, arguments)

    print(f"함수 실행: {function_name}({arguments}) -> {result}")

    function_outputs2.append(
        {
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps({"result": result}, ensure_ascii=False)
        }
    )

final_response2 = client.responses.create(
    model="gpt-4.1",
    previous_response_id=response2.id,
    input=function_outputs2,
    tools=tools,
)

print("\n=== 최종 답변 ===")
print(final_response2.output_text)


## 8) 왜 이게 MCP 이해에 중요한가?

지금은 함수가 **로컬 Python 코드**입니다.

- `get_weather(...)`
- `write_opening_line(...)`

하지만 MCP를 붙이면, 이 함수들이 **외부 MCP 서버가 제공하는 tool**로 바뀝니다.

그럼에도 구조는 거의 같습니다.

### 지금 예제
- 모델이 function call 반환
- 애플리케이션이 실제 함수 실행
- 결과를 다시 모델에 전달

### MCP 예제
- 모델/에이전트가 tool call 생성
- 애플리케이션이 MCP 서버 tool 실행
- 결과를 다시 모델/에이전트 흐름에 반영

즉, **OpenAI Function Calling은 MCP tool 호출을 이해하기 위한 가장 직접적인 출발점**입니다.


## 9) 실습 정리

이번 실습에서 확인한 핵심:

1. OpenAI Function Calling은 `tools`에 함수 schema를 넣는 방식입니다.
2. 모델은 함수를 직접 실행하지 않고, **function_call 요청**만 합니다.
3. 실제 함수 실행은 애플리케이션 코드가 합니다.
4. 실행 결과는 `function_call_output`으로 다시 모델에 전달합니다.
5. 이 구조는 나중에 MCP tool 호출 구조와 자연스럽게 연결됩니다.

다음 단계로 확장하려면:
- 로컬 함수 대신 **FastMCP 서버 tool**을 두고
- LangChain MCP client 또는 직접 MCP client로 연결하는 예제로 넘어가면 됩니다.
